# Notebook 02 — Secure Sandbox v3: Resource Control & Safety Filter
**Nhóm 67 | Tuần 4 | Ngôn ngữ Lập trình Python**

Notebook này kiểm thử hoạt động của **Secure Sandbox v3** (`src/runner_v3.py`) bao gồm:
1. **AST Whitelist check**: Chặn đứng các import và từ khóa nguy hiểm.
2. **Resource limitation**: Timeout (1.0s) và Memory limit (128MB bằng `psutil`).
3. **Docker Sandbox / Simulated Docker Sandbox**: Thực thi cách ly hoàn toàn.
4. **Detailed Exception / Vietnamese Feedback**: Diễn giải lỗi kỹ thuật sang tiếng Việt gợi ý sư phạm.

## Bước 1 — Import và Setup

In [1]:
import os
import sys
from pathlib import Path

from pathlib import Path
BASE = next((p for p in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents) if (p / 'data').exists() or (p / 'app.py').exists()), Path.cwd().resolve())

sys.path.insert(0, str(BASE / 'src'))
os.chdir(BASE)

from runner_v3 import grade_submission
from feedback import generate_vietnamese_feedback
print("✓ Import Secure Sandbox v3 thành công!")

✓ Import Secure Sandbox v3 thành công!


## Bước 2 — Kiểm tra Bộ lọc Tĩnh AST (Whitelist)

In [2]:
# Code nguy hiểm chứa import cấm (os)
banned_import_code = """
import os
def hack_system(x):
    return os.listdir('.')
"""

# Code nguy hiểm sử dụng exec
banned_exec_code = """
def eval_hack(x):
    exec(\"print('hack')\")
    return x
"""

tests = [{"input": "1", "expected": "1"}]

print("--- TEST 1: Chặn import cấm ---")
r1 = grade_submission(banned_import_code, "hack_system", tests)
print(f"Syntax OK: {r1['syntax_ok']}")
print(f"Lỗi: {r1['test_results'][0]['error_msg']}")

print("\n--- TEST 2: Chặn exec/eval ---")
r2 = grade_submission(banned_exec_code, "eval_hack", tests)
print(f"Syntax OK: {r2['syntax_ok']}")
print(f"Lỗi: {r2['test_results'][0]['error_msg']}")

--- TEST 1: Chặn import cấm ---
Syntax OK: False
Lỗi: Security Violation: Import cấm 'os' không có trong whitelist (Dòng 2)

--- TEST 2: Chặn exec/eval ---
Syntax OK: False
Lỗi: Security Violation: Hàm/Từ khóa bị cấm 'exec' (Dòng 3)


## Bước 3 — Kiểm tra Giới hạn Tài nguyên (TLE & MLE)

In [3]:
# Payload chạy quá thời gian (TLE)
tle_code = """
def infinite_loop(x):
    while True:
        pass
"""

# Payload cấp phát quá bộ nhớ (MLE)
mle_code = """
def memory_leak(x):
    y = ' ' * (200 * 1024 * 1024) # Cấp phát 200MB RAM
    return len(y)
"""

tests = [{"input": "1", "expected": "1"}]

print("--- TEST 3: Giới hạn thời gian (TLE) ---")
r3 = grade_submission(tle_code, "infinite_loop", tests)
print(f"Kết quả: {r3['test_results'][0]['status']}")
print(f"Chi tiết: {r3['test_results'][0]['error_msg']}")

print("\n--- TEST 4: Giới hạn bộ nhớ (MLE) ---")
r4 = grade_submission(mle_code, "memory_leak", tests)
print(f"Kết quả: {r4['test_results'][0]['status']}")
print(f"Chi tiết: {r4['test_results'][0]['error_msg']}")

--- TEST 3: Giới hạn thời gian (TLE) ---
Kết quả: TLE
Chi tiết: Time Limit Exceeded (>5.0s)

--- TEST 4: Giới hạn bộ nhớ (MLE) ---
Kết quả: MLE
Chi tiết: Memory Limit Exceeded (>128MB)


## Bước 4 — Kiểm tra Trích xuất Exception & Phản hồi Tiếng Việt

In [ ]:
# Code bị lỗi ZeroDivisionError
zero_div_code = """
def average(lst):
    return sum(lst) / len(lst)
"""

# Chạy test với list rỗng để kích hoạt chia cho 0
tests = [{"input": "[]", "expected": "None"}]
r_div = grade_submission(zero_div_code, "average", tests)
test_res = r_div['test_results'][0]

print("--- TEST 5: Phản hồi sư phạm tiếng Việt ---")
print(f"Lớp lỗi nhận diện: {test_res['status']}")
print(f"Traceback gốc:\n{test_res['error_msg']}\n")

# Tạo phản hồi gợi ý sư phạm (dùng hàm trong feedback.py)
fb_msg = generate_vietnamese_feedback(r_div)
print("THÔNG ĐIỆP PHẢN HỒI GỬI HỌC SINH:")
print(fb_msg)

--- TEST 5: Phản hồi sư phạm tiếng Việt ---
Lớp lỗi nhận diện: ZeroDivisionError
Traceback gốc:
Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Temp\runner_ai3qxa8z\solution.py", line 8, in <module>
    result = average([])
  File "C:\Users\Admin\AppData\Local\Temp\runner_ai3qxa8z\solution.py", line 5, in average
    return sum(lst) / len(lst)
           ~~~~~~~~~^~~~~~~~~~
ZeroDivisionError: division by zero



NameError: name 'get_feedback_message' is not defined